# 04 — Evaluation, calibration and leakage

## What the numbers do and do not mean

The URL model scores near-perfect on this benchmark (F1 ~0.9995). That is **benchmark
separability on PhiUSIIL**, not evidence of broad real-world phishing detection.
PhiUSIIL contains many trivially separable lexical artifacts. Report it as such.

The leakage experiment below quantifies exactly how much of that is artifact.

In [ ]:
import sys
from pathlib import Path

BACKEND = Path(r"E:\UIU\TRIMESTER - 11\Computer Security\CS_Project\Backend")
DATASET = BACKEND.parent / "Dataset"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

SEED = 42
np.random.seed(SEED)

print("backend :", BACKEND)
print("dataset :", DATASET)
print("numpy   :", np.__version__)
print("pandas  :", pd.__version__)

In [ ]:
def show(title, obj):
    print(f"\n=== {title} ===")
    print(obj)

def hr(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

NOTEBOOK_DIR = BACKEND / "notebooks"
ARTIFACT_DIR = NOTEBOOK_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def save_artifact(name, obj):
    """Persist a JSON-serialisable object and echo where it landed."""
    path = ARTIFACT_DIR / name
    path.write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")
    print(f"saved -> {path}")
    return path

## 4.1 Load the trained checkpoint

In [ ]:
import torch
from training.train_url import load_config, resolve_paths, class_weights

CFG_PATH = BACKEND / "configs" / "dev.yaml"
cfg = load_config(CFG_PATH)
paths = resolve_paths(cfg, CFG_PATH)

NOTEBOOK_CKPT = ARTIFACT_DIR / "url_model_notebook.pt"
CLI_CKPT = paths["checkpoints_dir"] / "url_model.pt"
CKPT = NOTEBOOK_CKPT if NOTEBOOK_CKPT.exists() else CLI_CKPT

print(f"using checkpoint: {CKPT}")
blob = torch.load(CKPT, map_location="cpu", weights_only=False)
show("checkpoint keys", list(blob.keys()))

# The CLI checkpoint written by training/train_url.py stores weights under
# "model_state"; the notebook checkpoint uses "state_dict". Accept either so this
# cell works after running notebook 03 or against the shipped checkpoint.
WEIGHTS_KEY = "state_dict" if "state_dict" in blob else "model_state"
TOK_KEY = "tokenizer_state" if "tokenizer_state" in blob else "tokenizer"
SCALER_KEY = "scaler_state" if "scaler_state" in blob else "scaler"
print(f"weights under '{WEIGHTS_KEY}', tokenizer under '{TOK_KEY}', scaler under '{SCALER_KEY}'")

mcfg = blob["model_config"]
print("model_config:", mcfg)

## 4.2 Rebuild and evaluate

In [ ]:
from app.models.url_model import URLCharModel
from app.preprocessing.url_dataset import load_splits, URLDataset, make_loader
from app.preprocessing.url_preprocessing import CharTokenizer
from app.preprocessing.url_features import FeatureScaler

mc = mcfg
MC = dict(mcfg)
MC.pop("dropout", None)  # not a constructor kwarg; config() omits it
model = URLCharModel(**MC)
model.load_state_dict(blob[WEIGHTS_KEY])
model.eval()

tok = CharTokenizer.from_state_dict(blob[TOK_KEY])
scaler = FeatureScaler.from_state_dict(blob[SCALER_KEY])
show("rebuilt", {
    "vocab_size": model.vocab_size if hasattr(model, "vocab_size") else MC.get("vocab_size"),
    "max_length": MC.get("max_length"),
    "handcrafted_active": model.use_handcrafted,
    "n_handcrafted": MC.get("n_handcrafted"),
})
if not model.use_handcrafted:
    print("NOTE: this checkpoint was trained WITHOUT the handcrafted branch.")

In [ ]:
splits = load_splits(paths["splits_dir"])

def predict(ds, batch=512):
    logits = []
    with torch.no_grad():
        for cb, mb, hb, _ in make_loader(ds, batch, shuffle=False):
            logits.append(model(cb, mb, hb).logit)
    return torch.cat(logits).numpy()

from app.utils.metrics import compute_binary_metrics

results = {}
test_metrics = None
for name in ("val", "test"):
    sp = splits[name]
    ds = URLDataset(sp, tok, scaler)
    logits = predict(ds)
    from scipy.special import expit
    results[name] = {"y": sp.labels, "logit": logits, "prob": expit(logits)}
    if name == "test":
        test_metrics = compute_binary_metrics(results["test"]["y"], results["test"]["prob"])
    print(f"{name}: n={len(sp):,}")

## 4.3 Metrics

In [ ]:
from app.utils.metrics import compute_binary_metrics

for name in ("val", "test"):
    r = results[name]
    m = compute_binary_metrics(r["y"], r["prob"], threshold=0.5)
    d = m.to_dict()
    hr(f"{name.upper()} metrics")
    show("headline", {
        "accuracy":  round(d["accuracy"], 6),
        "precision": round(d["precision"], 6),
        "recall":    round(d["recall"], 6),
        "f1":        round(d["f1"], 6),
        "specificity": round(d["specificity"], 6),
        "roc_auc":   round(d["roc_auc"], 6),
        "pr_auc":    round(d["pr_auc"], 6),
        "mcc":       round(d["mcc"], 6),
        "ece":       round(d["ece"], 6),
    })
    show("confusion", d["confusion_matrix"])

## 4.4 Confidence intervals (bootstrap)

In [ ]:
from app.utils.metrics import bootstrap_metric_ci

ev = cfg["evaluation"]
ci_rows = {}
for metric in ("recall", "precision", "f1"):
    ci = bootstrap_metric_ci(
        results["test"]["y"], results["test"]["prob"], metric=metric,
        threshold=0.5,
        n_resamples=min(200, ev.get("bootstrap_samples", 200)),
        confidence=ev.get("bootstrap_confidence", 0.95),
        seed=SEED,
    )
    ci_rows[metric] = ci
    print(f"{metric:>9}: {ci['low']:.4f} .. {ci['high']:.4f}")

show("recall at fixed FPR (test)", test_metrics.recall_at_fpr)

## 4.5 Leakage experiment

The central question: does the model exploit something that will not exist at
inference time? Domain-grouped splitting is the honest protocol. Compare it against
a naive random row-level split, where the same registered domain can appear in both
train and test.

In [ ]:
from app.preprocessing.url_features import registered_domain_of
from app.preprocessing.url_preprocessing import host_of

rows = []
for split_name in ("train", "val", "test"):
    sp = splits[split_name]
    for u, y in zip(sp.urls, sp.labels):
        rows.append({"split": split_name, "url": u, "label": int(y),
                     "reg": registered_domain_of(host_of(u))})

ldf = pd.DataFrame(rows)
show("rows", len(ldf))

groups = ldf.groupby("reg")["split"].nunique()
show("domains present in >1 split (grouped protocol)", int((groups > 1).sum()))
print("This must be 0. A non-zero count means the split is leaking.")

In [ ]:
hr("label noise check: hosts with both labels")
g = ldf.assign(host=ldf["url"].map(host_of)).groupby("host")["label"].nunique()
show("hosts carrying both labels", int((g > 1).sum()))
print("PhiUSIIL label noise puts a ceiling on achievable real-world performance.")

## 4.6 Mask-only baseline — read this before reporting fusion

In [ ]:
import pandas as pd

MANIFEST = BACKEND / "data" / "manifest.csv"
if MANIFEST.exists():
    mf = pd.read_csv(MANIFEST)
    avail = (mf["label"] == 1).astype(int)
    from sklearn.metrics import f1_score, precision_score, recall_score
    print("html_ok alone as a classifier:")
    print(f"  F1        {f1_score(avail, mf['html_ok'].astype(int)):.4f}")
    print(f"  precision {precision_score(avail, mf['html_ok'].astype(int)):.4f}")
    print(f"  recall    {recall_score(avail, mf['html_ok'].astype(int)):.4f}")

    rate = mf.groupby("label")["html_ok"].mean()
    print(f"\navailability gap (phishing - legitimate) = {rate.get(1,0) - rate.get(0,0):+.4f}")

    print("""
Modality availability is confounded with the label. A classifier that only reads
the availability mask already scores F1 ~0.834. Any fusion gain must be reported
against this baseline, not against chance.""")
else:
    print("manifest.csv not found — run the collector first.")

## 4.7 Save

In [ ]:
summary = {
    "url_model_test": {
        k: test_metrics.to_dict()[k]
        for k in ("accuracy", "precision", "recall", "f1", "specificity", "roc_auc", "pr_auc", "mcc")
    },
    "bootstrap_ci": ci_rows,
    "seed": SEED,
    "caveat": "Benchmark separability on PhiUSIIL; not a real-world generalisation claim.",
}
save_artifact("url_evaluation_notebook.json", summary)

## Next

`05_html_vision_preprocessing.ipynb` — HTML tokens, 45 features, screenshots.